##  Phase 0:- gita ingestion pipeline

#  Import All Libraries

In [1]:
import pandas as pd
import numpy as np
from langchain_community.document_loaders import PyPDFLoader
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_huggingface import ChatHuggingFace,HuggingFaceEmbeddings
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.messages import HumanMessage, AIMessage, BaseMessage
from langchain_core.output_parsers import StrOutputParser
from langchain_text_splitters import RecursiveCharacterTextSplitter

C:\Users\satya\AppData\Local\Temp\ipykernel_14036\3347508851.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


In [2]:
df=pd.read_csv(r'D:\d_drive_project\Generayive Ai Project\Sanatan_dharma_chatbot\Bhagwad_Gita.csv')


In [3]:
import pandas as pd
df = pd.read_csv("Bhagwad_Gita.csv")
print(df.shape)
print(df.columns.tolist())
df.head()

(701, 8)
['ID', 'Chapter', 'Verse', 'Shloka', 'Transliteration', 'HinMeaning', 'EngMeaning', 'WordMeaning']


,ID,Chapter,Verse,Shloka,Transliteration,HinMeaning,EngMeaning,WordMeaning
0,BG1.1,1,1,धृतराष्ट्र उवाच |\nधर्मक्षेत्रे कुरुक्षेत्रे स...,dhṛtarāṣṭra uvāca .\ndharmakṣetre kurukṣetre s...,।।1.1।।धृतराष्ट्र ने कहा -- हे संजय ! धर्मभूमि...,1.1 Dhritarashtra said What did my people and...,1.1 धर्मक्षेत्रे on the holy plain? कुरुक्षेत्...
1,BG1.2,1,2,सञ्जय उवाच |\nदृष्ट्वा तु पाण्डवानीकं व्यूढं द...,sañjaya uvāca .\ndṛṣṭvā tu pāṇḍavānīkaṃ vyūḍha...,।।1.2।।संजय ने कहा -- पाण्डव-सैन्य की व्यूह रच...,1.2. Sanjaya said Having seen the army of the...,1.2 दृष्ट्वा having seen? तु indeed? पाण्डवानी...
2,BG1.3,1,3,पश्यैतां पाण्डुपुत्राणामाचार्य महतीं चमूम् |\n...,paśyaitāṃ pāṇḍuputrāṇāmācārya mahatīṃ camūm .\...,।।1.3।।हे आचार्य ! आपके बुद्धिमान शिष्य द्रुपद...,"1.3. ""Behold, O Teacher! this mighty army of t...",1.3 पश्य behold? एताम् this? पाण्डुपुत्राणाम् ...
3,BG1.4,1,4,अत्र शूरा महेष्वासा भीमार्जुनसमा युधि |\nयुयुध...,atra śūrā maheṣvāsā bhīmārjunasamā yudhi .\nyu...,।।1.4।।इस सेना में महान् धनुर्धारी शूर योद्धा ...,"1.4. Here are heroes, mighty archers, eal in b...",1.4 अत्र here? शूराः heroes? महेष्वासाः mighty...
4,BG1.5,1,5,धृष्टकेतुश्चेकितानः काशिराजश्च वीर्यवान् |\nपु...,dhṛṣṭaketuścekitānaḥ kāśirājaśca vīryavān .\np...,"।।1.5।।धृष्टकेतु, चेकितान, बलवान काशिराज, पुर...","1.5. ""Dhrishtaketu, chekitana and the valiant ...",1.5 धृष्टकेतुः Dhrishtaketu? चेकितानः Chekitan...


In [4]:
records = []
for _, row in df.iterrows():
    records.append({
        "text": str(row["EngMeaning"]),
        "source": "Bhagavad Gita",
        "reference": f"Chapter {row['Chapter']}, Verse {row['Verse']}",
        "id": row["ID"],
        "shloka": str(row["Shloka"]),
        "hindi_meaning": str(row["HinMeaning"])
    })

print(len(records))
print(records[0])

701
{'text': '1.1 Dhritarashtra said  What did my people and the sons of Pandu do when they had assembled\ntogether eager for battle on the holy plain of Kurukshetra, O Sanjaya.', 'source': 'Bhagavad Gita', 'reference': 'Chapter 1, Verse 1', 'id': 'BG1.1', 'shloka': 'धृतराष्ट्र उवाच |\nधर्मक्षेत्रे कुरुक्षेत्रे समवेता युयुत्सवः |\nमामकाः पाण्डवाश्चैव किमकुर्वत सञ्जय ||१-१||', 'hindi_meaning': '।।1.1।।धृतराष्ट्र ने कहा -- हे संजय ! धर्मभूमि कुरुक्षेत्र में एकत्र हुए युद्ध के इच्छुक (युयुत्सव:) मेरे और पाण्डु के पुत्रों ने क्या किया?'}


In [5]:
from langchain_huggingface import HuggingFaceEmbeddings
embeddings=HuggingFaceEmbeddings(
    model_name="BAAI/bge-small-en-v1.5"
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [6]:
from langchain_community.vectorstores import FAISS

texts = [r["text"] for r in records]
metadatas = [
    {
        "source": r["source"],
        "reference": r["reference"],
        "id": r["id"],
        "shloka": r["shloka"],
        "hindi_meaning": r["hindi_meaning"]
    }
    for r in records
]




# Store in faissa
import os
from langchain_community.vectorstores import FAISS

db_path = r"D:\d_drive_project\Generayive Ai Project\Sanatan_dharma_chatbot\gita_faiss_index"

if os.path.exists(db_path) and len(os.listdir(db_path)) > 0:
    print("Loading existing vector store...")
    vector_store = FAISS.load_local(
        db_path,
        embeddings=embeddings,
        allow_dangerous_deserialization=True
    )
else:
    print("Creating new vector store...")
    vector_store = FAISS.from_texts(
        texts=texts,
        embedding=embeddings,
        metadatas=metadatas
    )
    vector_store.save_local(db_path)



Loading existing vector store...


In [7]:
# results = vector_store.similarity_search("What does Krishna say about duty?", k=3)
# for r in results:
#     print(r.metadata["reference"])
#     print("EN:", r.page_content[:150])
#     print("HI:", r.metadata["hindi_meaning"][:150])
#     print("---")

In [8]:
# print(df.duplicated(subset=["EngMeaning"]).sum())
# df[df["Chapter"] == 1][["Verse", "EngMeaning"]].iloc[18:24]

In [9]:
# results = vector_store.similarity_search("performing one's duty without attachment to results", k=5)
# for r in results:
#     print(r.metadata["reference"], "-", r.page_content[:100])

In [10]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_groq import ChatGroq
from dotenv import load_dotenv
import os
# llm=ChatGoogleGenerativeAI(model='gemini-2.5-flash')
llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0.3)
retriever=vector_store.as_retriever(search_kwargs={"k": 4})

def ask_gita(question):
    docs = retriever.invoke(question)
    context = "\n\n".join([
        f"[{d.metadata['reference']}]: {d.page_content}" for d in docs
    ])
    
    prompt = f"""You are a respectful guide to the Bhagavad Gita.
    Answer ONLY using the scripture excerpts below. Always cite the chapter and verse.
    If the excerpts don't fully answer the question, say so honestly.

    Excerpts:
    {context}

    Question: {question}

    Answer:"""
    
    response = llm.invoke(prompt)
    return response.content, [d.metadata['reference'] for d in docs]

In [11]:
# answer, sources = ask_gita("What does Krishna say about performing duty without attachment to results?")
# print(answer)
# print("\nSources:", sources)

In [12]:
# answer, sources = ask_gita("What is Krishna's favorite color?")
# print(answer)
# print("\nSources:", sources)

In [13]:
# answer, sources =ask_gita("What does Krishna say about the nature of the soul (Atman)?")
# print(answer)
# print("\nSources:", sources)

In [14]:
# answer, sources =ask_gita("What is the significance of the phrase 'Yoga is skill in action'?")
# print(answer)
# print("\nSources:", sources)

## Phase 1

In [15]:
# from langchain_community.document_loaders import PyPDFLoader

# loader = PyPDFLoader(r"D:\d_drive_project\Generayive Ai Project\Sanatan_dharma_chatbot\Data\Four-Vedas-English-Translation.pdf")
# pages = loader.load()

# print("Total pages:", len(pages))
# print(pages[1000].page_content[15000:])  # sample from somewhere in the middle, not the cover page

In [16]:
# import os

# pdf_folder = r"D:\d_drive_project\Generayive Ai Project\Sanatan_dharma_chatbot\Data"
# for fname in os.listdir(pdf_folder):
#     if fname.endswith(".pdf"):
#         loader = PyPDFLoader(os.path.join(pdf_folder, fname))
#         pages = loader.load()
        # print(fname, "->", len(pages), "pages")

In [17]:
# print(pages[300].page_content[:1500])
# print("\n\n---LATER PAGE---\n\n")
# print(pages[600].page_content[:1500])

In [18]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

def load_and_chunk_pdf(path, source_name, skip_first_n_pages=0):
    loader = PyPDFLoader(path)
    pages = loader.load()[skip_first_n_pages:]  # skip TOC/front matter
    
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=1000,
        chunk_overlap=150,
        separators=["\n\n", "\n", ". ", " "]
    )
    
    chunks = []
    for page in pages:
        page_num = page.metadata.get("page", "?") + 1  # 0-indexed -> human-readable
        splits = splitter.split_text(page.page_content)
        for chunk_text in splits:
            chunks.append({
                "text": chunk_text,
                "source": source_name,
                "reference": f"{source_name}, p.{page_num}",
                "page": page_num
            })
    return chunks

In [ ]:
base = r"D:\d_drive_project\Generayive Ai Project\Sanatan_dharma_chatbot\Data"

vedas_chunks = load_and_chunk_pdf(f"{base}\\Four-Vedas-English-Translation.pdf", "Four Vedas", skip_first_n_pages=50)
upanishads_chunks = load_and_chunk_pdf(f"{base}\\108upanishads.pdf", "108 Upanishads", skip_first_n_pages=10)
puranas_chunks = load_and_chunk_pdf(f"{base}\\18 Puranas.pdf", "18 Puranas", skip_first_n_pages=10)
bhagavatam_chunks = load_and_chunk_pdf(f"{base}\\srimad-bhagavata-mahapurana-english-translations.pdf", "Srimad Bhagavatam", skip_first_n_pages=10)

print(len(vedas_chunks), len(upanishads_chunks), len(puranas_chunks), len(bhagavatam_chunks))

In [ ]:
all_chunks = vedas_chunks + upanishads_chunks + puranas_chunks + bhagavatam_chunks
print("all_chunks rebuilt:", len(all_chunks))

all_chunks rebuilt: 17624


In [ ]:
import json

with open("mahabharata_progress.json", "r", encoding="utf-8") as f:
    all_mahabharata_chunks = json.load(f)

print("Mahabharata chunks loaded:", len(all_mahabharata_chunks))

Mahabharata chunks loaded: 2108


In [ ]:
# import torch
# print(torch.cuda.is_available())
# print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU detected")

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-small-en-v1.5",
    model_kwargs={"device": "cuda"},
    encode_kwargs={"normalize_embeddings": True, "batch_size": 64}
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [ ]:
# from langchain_community.vectorstores import FAISS
# import time

# db_path = r"D:\d_drive_project\Generayive Ai Project\Sanatan_dharma_chatbot\scripture_faiss_index"

# batch_size = 500
# scripture_vectorstore = None

# start = time.time()

# # First: all_chunks (Vedas, Upanishads, Puranas, Bhagavatam)
# for i in range(0, len(all_chunks), batch_size):
#     batch = all_chunks[i:i+batch_size]
#     texts = [c["text"] for c in batch]
#     metadatas = [{"source": c["source"], "reference": c["reference"], "page": c["page"]} for c in batch]

#     if scripture_vectorstore is None:
#         scripture_vectorstore = FAISS.from_texts(texts=texts, embedding=embeddings, metadatas=metadatas)
#     else:
#         scripture_vectorstore.add_texts(texts=texts, metadatas=metadatas)

#     elapsed = time.time() - start
#     print(f"[all_chunks] {min(i+batch_size, len(all_chunks))}/{len(all_chunks)} — {elapsed/60:.1f} min")

# # Then: Mahabharata chunks
# for i in range(0, len(all_mahabharata_chunks), batch_size):
#     batch = all_mahabharata_chunks[i:i+batch_size]
#     texts = [c["text"] for c in batch]
#     metadatas = [{"source": c["source"], "reference": c["reference"], "book_num": c["book_num"]} for c in batch]
#     scripture_vectorstore.add_texts(texts=texts, metadatas=metadatas)
#     elapsed = time.time() - start
#     print(f"[mahabharata] {min(i+batch_size, len(all_mahabharata_chunks))}/{len(all_mahabharata_chunks)} — {elapsed/60:.1f} min")

# print("\nFinal total vectors:", scripture_vectorstore.index.ntotal)
# scripture_vectorstore.save_local(db_path)
# print("Saved to", db_path)

In [ ]:
import os
from langchain_community.vectorstores import FAISS
import time

db_path = r"D:\d_drive_project\Generayive Ai Project\Sanatan_dharma_chatbot\scripture_faiss_index"

if os.path.exists(db_path) and len(os.listdir(db_path)) > 0:
    print("Loading existing scripture vector store...")
    scripture_vectorstore = FAISS.load_local(
        db_path,
        embeddings=embeddings,
        allow_dangerous_deserialization=True
    )
    print("Loaded. Total vectors:", scripture_vectorstore.index.ntotal)

else:
    print("Creating new scripture vector store...")
    all_chunks = vedas_chunks + upanishads_chunks + puranas_chunks + bhagavatam_chunks
    print("Total chunks to embed:", len(all_chunks))

    batch_size = 500
    scripture_vectorstore = None

    start = time.time()
    for i in range(0, len(all_chunks), batch_size):
        batch = all_chunks[i:i+batch_size]
        texts = [c["text"] for c in batch]
        metadatas = [{"source": c["source"], "reference": c["reference"], "page": c["page"]} for c in batch]

        if scripture_vectorstore is None:
            scripture_vectorstore = FAISS.from_texts(texts=texts, embedding=embeddings, metadatas=metadatas)
        else:
            scripture_vectorstore.add_texts(texts=texts, metadatas=metadatas)

        elapsed = time.time() - start
        print(f"Processed {min(i+batch_size, len(all_chunks))}/{len(all_chunks)} — {elapsed/60:.1f} min elapsed")

    scripture_vectorstore.save_local(db_path)
    print("Done. Saved to", db_path)

Loading existing scripture vector store...
Loaded. Total vectors: 19732


In [ ]:
scripture_retriever = scripture_vectorstore.as_retriever(search_kwargs={"k": 5})

results = scripture_retriever.invoke("What does scripture say about performing one's duty?")
for r in results:
    print(r.metadata["source"], "-", r.metadata["reference"])
    print(r.page_content[:200])
    print("---")

Four Vedas - Four Vedas, p.894
He who attaineth their service shall live. 
l O Agni, lead. 
m Of the gods. 
n May they be prosperous for us 
 
o In every contest. 
In the waters, O Agni, is thy seat, 
Thou enterest the plants; 
Bei
---
108 Upanishads - 108 Upanishads, p.213
Do your duty. Dharma–here translated “duty”–is the way of life in accordance with the deep 
wellsprings of our personality–karma and samskaras. These comprise our fundamental nature, our 
prakriti. Th
---
108 Upanishads - 108 Upanishads, p.210
“Let your conduct be marked by right action, including study and teaching of the scriptures; by 
truthfulness in word, deed, and thought; by self-denial and the practice of austerity; by poise 
and se
---
Four Vedas - Four Vedas, p.1350
1. From near thy vicinity, from near thy distance (do I call): remain here, do not follow; do not 
follow the Fathers of yore! Firmly do I fasten thy life's breath. 
2. Whatever sorcery any kinsman or
---
Srimad Bhagavatam - Srimad Bhagavatam, 

In [ ]:
gita_retriever = vector_store.as_retriever(search_kwargs={"k": 3})
scripture_retriever = scripture_vectorstore.as_retriever(search_kwargs={"k": 5})

In [ ]:
def ask_sanatan(question, org_name="Sanatan Dharma Chatbot"):
    gita_docs = gita_retriever.invoke(question)
    scripture_docs = scripture_retriever.invoke(question)

    gita_context = "\n\n".join([
        f"[Bhagavad Gita, {d.metadata['reference']}]: {d.page_content}"
        for d in gita_docs
    ])

    scripture_context = "\n\n".join([
        f"[{d.metadata['reference']}]: {d.page_content}"   # removed d.metadata['source'] prefix
        for d in scripture_docs
    ])

    prompt = f"""You are a respectful scripture-grounded guide for {org_name}.
        Answer using ONLY the excerpts below. Never invent a verse or claim not present in the excerpts.

        --- BHAGAVAD GITA EXCERPTS ---
        {gita_context}

        --- OTHER SCRIPTURE EXCERPTS (Vedas, Upanishads, Puranas, Srimad Bhagavatam) ---
        {scripture_context}

        Question: {question}

        Structure your answer exactly as follows:

        1. Direct Answer: (2-3 sentences answering the question plainly)

        2. Bhagavad Gita Guidance: List each Gita excerpt above that is genuinely relevant, with citation.
        If NONE of the Gita excerpts are relevant to the question, write exactly: "No directly relevant verse found in the retrieved excerpts." and do not cite any Gita verse in this section.
        Do not mix both — either cite relevant verses, or say none were found, never both.

        3. Supporting Teachings from Other Scriptures: Same rule — list genuinely relevant excerpts with citation, or say none were found. Never both.

        4. Explanation: (2-3 sentences connecting the above teachings to the question, in plain language)
        """

    response = llm.invoke(prompt)
    all_sources = (
        [f"Bhagavad Gita, {d.metadata['reference']}" for d in gita_docs] +
        [d.metadata['reference'] for d in scripture_docs]   # already has source name baked in
    )
    return response.content, all_sources

In [ ]:
# answer, sources = ask_sanatan("Is playing sports good or bad according to Sanatan Dharma?")
# print(answer)
# print("\n\nAll retrieved sources:", sources)

In [ ]:
# answer, sources = ask_sanatan("What does Sanatan Dharma say about controlling anger?")
# print(answer)
# print("\nSources:", sources)

In [ ]:
# answer, sources = ask_sanatan("What does the Gita say about eating meat?")
# print(answer)
# print("\nSources:", sources)

In [ ]:
# results = scripture_vectorstore.similarity_search("Why did Arjuna hesitate before the battle?", k=5)
# for r in results:
#     print(r.metadata["source"], "-", r.metadata["reference"])
#     print(r.page_content[:150])
#     print("---")

In [ ]:
# answer, sources = ask_sanatan("Why did Arjuna hesitate before the battle at Kurukshetra?")
# print(answer)
# print("\nSources:", sources)

##  Step 2: Build a BM25 retriever alongside your existing FAISS one

In [ ]:
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever
# Build BM25 from the same texts already in your scripture index
scripture_texts_for_bm25 = [c["text"] for c in all_chunks] + [c["text"] for c in all_mahabharata_chunks]
scripture_metadatas_for_bm25 = (
    [{"source": c["source"], "reference": c["reference"]} for c in all_chunks] +
    [{"source": c["source"], "reference": c["reference"], "book_num": c["book_num"]} for c in all_mahabharata_chunks]
)

from langchain_core.documents import Document
bm25_docs = [Document(page_content=t, metadata=m) for t, m in zip(scripture_texts_for_bm25, scripture_metadatas_for_bm25)]

bm25_retriever = BM25Retriever.from_documents(bm25_docs)
bm25_retriever.k = 8

vector_retriever = scripture_vectorstore.as_retriever(search_kwargs={"k": 8})

ensemble_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, vector_retriever],
    weights=[0.4, 0.6]   # slightly favor semantic, but let keyword matches pull their weight
)
##  Step 1: Multi-Query Retrieval
from langchain_classic.retrievers import MultiQueryRetriever

multiquery_retriever = MultiQueryRetriever.from_llm(
    retriever=ensemble_retriever,
    llm=llm
)


In [ ]:
# results = ensemble_retriever.invoke("Why did Arjuna hesitate before the battle?")
# for r in results[:8]:
#     print(r.metadata.get("source", "?"), "-", r.metadata.get("reference", "?"))
#     print(r.page_content[:150])
#     print("---")

In [ ]:
# results = ensemble_retriever.invoke("Why did Arjuna hesitate before the battle?")
# for r in results[:8]:
#     print(r.metadata.get("source", "?"), "-", r.metadata.get("reference", "?"))
#     print(r.page_content[:150])
#     print("---")

In [ ]:
# print(type(ensemble_retriever))
# print(ensemble_retriever.retrievers)
# print(ensemble_retriever.weights)

In [ ]:
# test_results = ensemble_retriever.invoke("Why did Arjuna hesitate before the battle?")
# print("Number of results:", len(test_results))
# for r in test_results[:8]:
#     print(r.metadata.get("reference", "?"))
#     print(r.page_content[:150])
#     print("---")

In [ ]:
# print("Number of results:", len(test_results))
# for i, r in enumerate(test_results):
#     print(f"{i+1}.", r.metadata.get("reference", "?"))
#     print(r.page_content[:150])
#     print("---")

In [ ]:
# results = ensemble_retriever.invoke("What does the Markandeya Purana say about the cycle of yugas?")
# for i, r in enumerate(results[:8]):
#     print(f"{i+1}.", r.metadata.get("reference", "?"))
#     print(r.page_content[:150])
#     print("---")

##  add the cross-encoder reranker

In [ ]:
from sentence_transformers import CrossEncoder

reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2", device="cuda")

def rerank(question, docs, top_k=5):
    pairs = [[question, d.page_content] for d in docs]
    scores = reranker.predict(pairs)
    scored_docs = sorted(zip(scores, docs), key=lambda x: x[0], reverse=True)
    return [doc for score, doc in scored_docs[:top_k]]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

In [ ]:
raw_results = ensemble_retriever.invoke("Why did Arjuna hesitate before the battle?")
reranked = rerank("Why did Arjuna hesitate before the battle?", raw_results, top_k=5)

for r in reranked:
    print(r.metadata.get("reference", "?"))
    print(r.page_content[:150])
    print("---")

Book 7, The Mahabharata, Book 7: Drona Parva: Dronabhisheka Parva...
SECTION XVII
"Sanjaya said, 'The troops of both the armies, having proceeded to their tents, duly took up their quarters, O king, according to the div
---
Book 7, The Mahabharata, Book 7: Drona Parva: Jayadratha-Vadha Pa...
SECTION LXXXIX
"Dhritarashtra said, 'When the van of my army thus slaughtered by the diadem-decked (Arjuna) broke and fled, who were those heroes that
---
Book 4, The Mahabharata, Book 4: Virata Parva: Go-harana Parva: S...
SECTION LXI
"Vaisampayana said, 'Having defeated Vikartana's son, Arjuna said unto the son of Virata, 'Take me towards that division where yonder devi
---
Book 1, The Mahabharata, Book 1: Adi Parva: Swayamvara Parva: Sec...
SECTION CLXLII
(Swayamvara Parva continued)
"Vaisampayana said, 'Then those bulls among Brahmanas shaking their deer-skins and water-pots made of coco
---
Book 7, The Mahabharata, Book 7: Drona Parva: Abhimanyu-badha Par...
SECTION XLII
"Sanjaya said, 'When t

In [ ]:
raw_results2 = ensemble_retriever.invoke("What hymns in the Rigveda praise Agni?")
reranked2 = rerank("What hymns in the Rigveda praise Agni?", raw_results2, top_k=5)

for r in reranked2:
    print(r.metadata.get("reference", "?"))
    print(r.page_content[:150])
    print("---")

Four Vedas, p.337
Rig Veda – English Translation 
2 Through his great might o'ercoming all misfortunes, praised in the house is Agni Jatavedas.  
May he protect us from
---
Four Vedas, p.258
Rig Veda – English Translation 
 
HYMN XVI. Agni.  
1. GREAT power is in the beam of light, sing praise to, Agni, to the God  
Whom men have set in fo
---
Four Vedas, p.257
Rig Veda – English Translation 
6 He who pays sacrifice to thee with homage, O Agni, keeps the Red Steer's Law eternal;  
Wide is his dwelling. May th
---
Four Vedas, p.396
Rig Veda – English Translation 
21 Grant us a home with triple guard, Aryaman, Mitra, Varuna!  
Unthreatened, Maruts! meet for praise, and filled with
---
Four Vedas, p.1185
Hymns of the Sama Veda - Translation - Griffith 
 
1184 
 
6. Worship the Vasus, Agni! here, the Rudras and Adityas, all 
Who know fair sacrifices, sp
---


In [ ]:
def ask_sanatan(question, org_name="Sanatan Dharma Chatbot"):
    gita_docs = gita_retriever.invoke(question)
    
    raw_scripture_docs = ensemble_retriever.invoke(question)
    scripture_docs = rerank(question, raw_scripture_docs, top_k=5)

    gita_context = "\n\n".join([
        f"[Bhagavad Gita, {d.metadata['reference']}]: {d.page_content}"
        for d in gita_docs
    ])

    scripture_context = "\n\n".join([
        f"[{d.metadata['reference']}]: {d.page_content}"
        for d in scripture_docs
    ])

    prompt = f"""You are a respectful scripture-grounded guide for {org_name}.
Answer using ONLY the excerpts below. Never invent a verse or claim not present in the excerpts.

--- BHAGAVAD GITA EXCERPTS ---
{gita_context}

--- OTHER SCRIPTURE EXCERPTS (Vedas, Upanishads, Puranas, Srimad Bhagavatam, Mahabharata) ---
{scripture_context}

Question: {question}

Structure your answer exactly as follows:

1. Direct Answer: (2-3 sentences answering the question plainly)

2. Bhagavad Gita Guidance: List each Gita excerpt above that is genuinely relevant, with citation.
If NONE of the Gita excerpts are relevant, write exactly: "No directly relevant verse found in the retrieved excerpts." Do not mix both.

3. Supporting Teachings from Other Scriptures: Same rule.

4. Explanation: (2-3 sentences connecting the above teachings to the question, in plain language)
"""

    response = llm.invoke(prompt)
    all_sources = (
        [f"Bhagavad Gita, {d.metadata['reference']}" for d in gita_docs] +
        [d.metadata['reference'] for d in scripture_docs]
    )
    return response.content, all_sources

In [ ]:
answer, sources = ask_sanatan("What does Sanatan Dharma say about controlling anger?")
print(answer)
print("\nSources:", sources)

1. Direct Answer: Sanatan Dharma emphasizes the importance of controlling anger, as it is considered a destructive emotion that can lead to harm for oneself and others. According to the teachings, one should strive to suppress anger and cultivate forgiveness. This is seen as a key aspect of achieving peace and prosperity.

2. Bhagavad Gita Guidance: 
- No directly relevant verse found in the retrieved excerpts.

3. Supporting Teachings from Other Scriptures: 
- [Book 3, The Mahabharata, Book 3: Vana Parva: Arjunabhigamana Parv...]: SECTION XXIX (entire section discusses the importance of controlling anger)
- [Srimad Bhagavatam, p.287]: verses 32-35 (warn against the dangers of anger and advise pacification)

4. Explanation: The teachings from the Mahabharata and Srimad Bhagavatam highlight the negative consequences of anger and the benefits of forgiveness, which aligns with the overall emphasis of Sanatan Dharma on cultivating self-control and inner peace. By recognizing the destructiv

In [ ]:
answer, sources = ask_sanatan("Why did Arjuna hesitate before the battle at Kurukshetra?")
print(answer)
print("\nSources:", sources)

1. Direct Answer: Arjuna hesitated before the battle at Kurukshetra because he was overwhelmed with sorrow and did not want to fight against his own kin. He cast away his bow and arrow and sat down on the seat of the chariot, indicating his reluctance to engage in the battle. This hesitation was rooted in his emotional struggle with the idea of fighting against his own family members.

2. Bhagavad Gita Guidance: 
- 1.47: Sanjaya said  Having thus spoken in the midst of the battlefield, Arjuna, casting away his bow and arrow, sat down on the seat of the chariot with his mind overwhelmed with sorrow.
- 1.22: Arjuna said  In the middle between the two armies, place my chariot, O krishna, so that I may behold those who stand here desirous to fight, and know with whom I must fight, when the battle is about to commence.
- 3.1: Arjuna said  If Thou thinkest that knowledge is superior to action, O Krishna, why then, O Kesava, dost Thou ask me to engage in this terrible action?

3. Supporting T

In [ ]:
answer, sources = ask_sanatan("Why did Arjuna hesitate before the battle at Kurukshetra?")
print(answer)
print("\nSources:", sources)

1. Direct Answer: Arjuna hesitated before the battle at Kurukshetra because he was overwhelmed with sorrow and did not want to fight against his own kin. He cast away his bow and arrow and sat down on the seat of the chariot, unable to proceed with the battle. This hesitation was rooted in his emotional and moral dilemma.

2. Bhagavad Gita Guidance: 
- 1.47: Sanjaya said  Having thus spoken in the midst of the battlefield, Arjuna, casting away his bow and arrow, sat down on the seat of the chariot with his mind overwhelmed with sorrow.
- 1.22: Arjuna said  In the middle between the two armies, place my chariot, O krishna, so that I may behold those who stand here desirous to fight, and know with whom I must fight, when the battle is about to commence.
- 3.1: Arjuna said  If Thou thinkest that knowledge is superior to action, O Krishna, why then, O Kesava, dost Thou ask me to engage in this terrible action?

3. Supporting Teachings from Other Scriptures: No directly relevant verse found

##  Step 1: Multi-Query Retrieval

In [ ]:
# test it
import logging
logging.basicConfig()
logging.getLogger("langchain.retrievers.multi_query").setLevel(logging.INFO)  # shows the generated query variants

results = multiquery_retriever.invoke("is sports ok")
print("Total unique results:", len(results))
for r in results[:8]:
    print(r.metadata.get("reference", "?"))
    print(r.page_content[:150])
    print("---")

Total unique results: 60
Srimad Bhagavatam, p.614
peaceful in all activities, for one is situated in eternal blissful life. Once situated on that 
platform, one does not return to materialistic activi
---
Srimad Bhagavatam, p.566
perceived with reference to the sense objects by contact with the body can be obtained in any 
form of life, according to one’s past fruitive activiti
---
Srimad Bhagavatam, p.527
ultimate goal of life. On the other hand, if one acts without desires for fruitive resultsin other 
words, if one engages in devotional activitieshe c
---
108 Upanishads, p.66
truth regarding spiritual life. 
Only the spirit is eternal and everlasting. Everything else, however highly evolved or sacred, is 
temporal and imper
---
Srimad Bhagavatam, p.481
is always transcendental to this material creation. If the members of human society do not 
understand Him, the Supreme, through their advancement in 
---
108 Upanishads, p.71
happiness is an inalienable right for every human being. B

In [ ]:
reranked = rerank("is sports ok", results, top_k=5)
for r in reranked:
    print(r.metadata.get("reference", "?"))
    print(r.page_content[:200])
    print("---")

Four Vedas, p.534
4 Others caress the wife of him whose riches the die hath coveted, that rapid courser:  
Of him speak father, mother, brothers saying, We know him not: bind him and take him with you.  
5 When I resol
---
Srimad Bhagavatam, p.1121
Srimad Bhagavata Mahapurana 
 
1121 
 
12Afterward, Lord Kṛṣṇa and His wives would give the ornaments and clothing they had worn 
during their water sports to the male and female performers, who earne
---
Book 18, The Mahabharata, Book 18: Svargarohanika Parva: Section 6
6
Janamejaya said, "O holy one, according to what rites should the learned listen to the Bharata? What are the fruits (acquirable by hearing it)? What deities are to be worshipped during the several p
---
Book 10, The Mahabharata, Book 10: Sauptika Parva: Section 2
2
Kripa said, "We have heard all that thou hast said, O puissant one! Listen, however, to a few words of mine, O mighty armed one! All men are subjected to and governed by these two forces, Destiny an
---
108 Upan

In [ ]:
results=scripture_vectorstore.similarity_search("table of content",k=20)
for r in results:
    print(r.metadata["reference"],"-",r.page_content[:100])

Srimad Bhagavatam, p.15 - 22.  Enumeration of the Elements of Material Creation …………………………………………………………………………………….1200 
23.  Th
Book 6 (Bhishma Parva), The Mahabharata, Book 6: Bhishma Parva: Jamvu-khanda Nirm... - SECTION V
"Dhritarashtra said,--'The names of rivers and mountains, O Sanjaya, as also of provinces,
108 Upanishads, p.436 - Sound is its externally correlated object element. The tongue is one portion taken out of it. Taste 
Four Vedas, p.1177 - a complete glossary, and explanatory notes; and in 1874-78 Pandit Satyavrata Samasrami of 
Calcutta 
Book 3, The Mahabharata, Book 3: Vana Parva: Markandeya-Samasya P... - SECTION CCX
"Markandeya continued, 'O Bharata, the Brahmana, thus interrogated by the virtuous fowle
Book 2, The Mahabharata, Book 2: Sabha Parva: Lokapala Sabhakhaya... - SECTION VIII
"Narada said,--'O Yudhisthira, I shall now describe the assembly house of Yama, the son
108 Upanishads, p.743 - 6- 7. The body with the seven constituents (Chile, blood, flesh, fat, 

In [ ]:
def ask_sanatan(question, org_name="Sanatan Dharma Chatbot"):
    gita_docs = gita_retriever.invoke(question)
    
    raw_scripture_docs = multiquery_retriever.invoke(question)
    scripture_docs = rerank(question, raw_scripture_docs, top_k=5)

    gita_context = "\n\n".join([
        f"[Bhagavad Gita, {d.metadata['reference']}]: {d.page_content}"
        for d in gita_docs
    ])

    scripture_context = "\n\n".join([
        f"[{d.metadata['reference']}]: {d.page_content}"
        for d in scripture_docs
    ])

    prompt = f"""You are a respectful scripture-grounded guide for {org_name}.
Answer using ONLY the excerpts below. Never invent a verse or claim not present in the excerpts.

--- BHAGAVAD GITA EXCERPTS ---
{gita_context}

--- OTHER SCRIPTURE EXCERPTS (Vedas, Upanishads, Puranas, Srimad Bhagavatam, Mahabharata) ---
{scripture_context}

Question: {question}

Structure your answer exactly as follows:

1. Direct Answer: (2-3 sentences answering the question plainly)

2. Bhagavad Gita Guidance: List each Gita excerpt above that is genuinely relevant, with citation.
If NONE of the Gita excerpts are relevant, write exactly: "No directly relevant verse found in the retrieved excerpts." Do not mix both.

3. Supporting Teachings from Other Scriptures: Same rule.

4. Explanation: (2-3 sentences connecting the above teachings to the question, in plain language)
"""

    response = llm.invoke(prompt)
    all_sources = (
        [f"Bhagavad Gita, {d.metadata['reference']}" for d in gita_docs] +
        [d.metadata['reference'] for d in scripture_docs]
    )
    return response.content, all_sources

In [ ]:
test_questions = [
    "Is playing sports good or bad according to Sanatan Dharma?",
    "What does Sanatan Dharma say about controlling anger?",
    "Why did Arjuna hesitate before the battle at Kurukshetra?",
    "What hymns in the Rigveda praise Agni?",
]

for q in test_questions:
    print("="*80)
    print("Q:", q)
    answer, sources = ask_sanatan(q)
    print(answer)
    print("\nSources:", sources)
    print()

Q: Is playing sports good or bad according to Sanatan Dharma?
1. Direct Answer: According to Sanatan Dharma, whether playing sports is good or bad depends on its impact on one's consciousness and pursuit of the Goal. If it helps in expanding consciousness and aids in the search for God, it is considered good. However, if it takes one away from the Goal or hinders spiritual growth, it is not beneficial.

2. Bhagavad Gita Guidance: No directly relevant verse found in the retrieved excerpts.

3. Supporting Teachings from Other Scriptures: 
- [108 Upanishads, p.91]: This excerpt discusses what is considered good or bad in the context of Sanatan Dharma, emphasizing the impact on consciousness and the pursuit of God.
- [108 Upanishads, p.23]: This excerpt highlights the importance of balancing worldly life with spiritual growth, which can be applied to the context of playing sports.

4. Explanation: The teachings from the 108 Upanishads suggest that activities like playing sports should be e